In [1]:
"""
seaad_inference.py

Loads pre-trained models from all 4 Morabito pipelines and runs inference
on the SEA-AD held-out dataset. Reports AUC per split and averaged,
per cell type, per model type.

Usage:
    python seaad_inference.py
"""

import os
import pandas as pd
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
SEAAD_META_PATH   = '/n/groups/patel/adithya/SEAAD_Outputs/SEAAD_CellMetadata.parquet'
SEAAD_MATRIX_DIR  = '/n/groups/patel/adithya/SEAAD_Outputs/'
RESULTS_OUT       = '/n/groups/patel/adithya/SEAAD_Outputs/inference_results.csv'

# 4 trained model directories — one per pipeline
MODEL_DIRS = {
    'genes_only'   : '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new/',
    'genes_apoe'   : '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_apoe_new/',
    'genes_demo'   : '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_both_new/',
    'demo_only'    : '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_demo_new/',
}

CELL_TYPES = ['Ex', 'Inh', 'Ast', 'Oli', 'Mic', 'Opc']
N_SPLITS   = 5  # adjust if you ran more or fewer splits


def load_seaad_matrix(cell_type):
    path = os.path.join(SEAAD_MATRIX_DIR, f'SEAAD_Matrix_{cell_type}.parquet')
    print(f"    Loading SEA-AD matrix for {cell_type}...")
    matrix = pd.read_parquet(path)
    # Clean column names to match how Morabito models were trained
    matrix.columns = matrix.columns.str.replace(r'[^A-Za-z0-9_]+', '', regex=True)
    matrix.index.name = 'TAG'
    return matrix


def get_seaad_X_y(matrix, metadata, cell_type):
    """Subset metadata and matrix to this cell type, return aligned X and y."""
    ct_meta = metadata[metadata['broad_cell_type'] == cell_type].copy()
    # Keep only cells present in both
    common_tags = matrix.index.intersection(ct_meta.index)
    X = matrix.loc[common_tags]
    y = ct_meta.loc[common_tags, 'alzheimers_or_control']
    return X, y


def run_inference_for_model(model_path, X_full, y, model_type, cell_type, split_index):
    """
    Load a single joblib model, subset X to its expected features,
    run predict_proba, return AUC and AUPRC.
    """
    if not os.path.exists(model_path):
        print(f"      MISSING: {model_path}")
        return None

    model = joblib.load(model_path)

    # Get features the model expects
    expected_features = list(model.feature_names_in_)

    # Intersect with what SEA-AD has
    available_features = [f for f in expected_features if f in X_full.columns]
    missing = len(expected_features) - len(available_features)

    if len(available_features) == 0:
        print(f"      ERROR: No overlapping features for {model_type} {cell_type} split {split_index}")
        return None

    if missing > 0:
        print(f"      WARNING: {missing}/{len(expected_features)} features missing from SEA-AD — using {len(available_features)} available")

    X_sub = X_full[available_features]

    # For demo_only model, features are metadata not genes —
    # SEA-AD matrix won't have these. Handle gracefully.
    try:
        y_prob = model.predict_proba(X_sub)[:, 1]
    except Exception as e:
        print(f"      ERROR during predict_proba: {e}")
        return None

    auc   = roc_auc_score(y, y_prob)
    auprc = average_precision_score(y, y_prob)

    return {
        'model_type'  : model_type,
        'cell_type'   : cell_type,
        'split'       : split_index,
        'n_cells'     : len(y),
        'n_cases'     : int(y.sum()),
        'n_controls'  : int((y == 0).sum()),
        'n_features_expected' : len(expected_features),
        'n_features_used'     : len(available_features),
        'roc_auc'     : round(auc, 4),
        'auprc'       : round(auprc, 4),
    }


def main():
    print("Loading SEA-AD metadata...")
    metadata = pd.read_parquet(SEAAD_META_PATH)
    print(f"  Metadata shape: {metadata.shape}")

    all_results = []

    for cell_type in CELL_TYPES:
        print(f"\n{'='*60}")
        print(f"Cell type: {cell_type}")

        # Load SEA-AD matrix for this cell type once
        try:
            matrix = load_seaad_matrix(cell_type)
        except FileNotFoundError:
            print(f"  MISSING matrix for {cell_type}, skipping.")
            continue

        X_full, y = get_seaad_X_y(matrix, metadata, cell_type)
        print(f"  SEA-AD cells: {len(y):,}  |  Cases: {int(y.sum())}  |  Controls: {int((y==0).sum())}")

        for model_type, model_dir in MODEL_DIRS.items():
            print(f"\n  Model: {model_type}")

            # Demo-only model has no gene features — skip gene matrix lookup
            # It will fail at predict_proba and be caught gracefully
            for split_idx in range(1, N_SPLITS + 1):
                model_path = os.path.join(
                    model_dir, cell_type, f'split_{split_idx}', 'maximal_classifier.joblib'
                )
                result = run_inference_for_model(
                    model_path, X_full, y, model_type, cell_type, split_idx
                )
                if result is not None:
                    all_results.append(result)
                    print(f"      Split {split_idx}: AUC={result['roc_auc']:.4f}  AUPRC={result['auprc']:.4f}")

    # ── Build results DataFrame ────────────────────────────────────────────────
    results_df = pd.DataFrame(all_results)

    # ── Compute averaged AUC per model_type x cell_type ───────────────────────
    avg_df = (
        results_df
        .groupby(['model_type', 'cell_type'])
        .agg(
            mean_roc_auc  = ('roc_auc', 'mean'),
            std_roc_auc   = ('roc_auc', 'std'),
            mean_auprc    = ('auprc', 'mean'),
            std_auprc     = ('auprc', 'std'),
            n_splits      = ('split', 'count'),
        )
        .round(4)
        .reset_index()
    )

    # ── Print summary ──────────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("SUMMARY — Mean AUC per model type and cell type")
    print('='*60)
    print(avg_df.to_string(index=False))

    # ── Save both per-split and averaged results ───────────────────────────────
    results_df.to_csv(RESULTS_OUT.replace('.csv', '_per_split.csv'), index=False)
    avg_df.to_csv(RESULTS_OUT.replace('.csv', '_averaged.csv'), index=False)

    print(f"\nSaved:")
    print(f"  Per-split : {RESULTS_OUT.replace('.csv', '_per_split.csv')}")
    print(f"  Averaged  : {RESULTS_OUT.replace('.csv', '_averaged.csv')}")


if __name__ == '__main__':
    main()

Loading SEA-AD metadata...
  Metadata shape: (1297754, 11)

Cell type: Ex
    Loading SEA-AD matrix for Ex...
  SEA-AD cells: 610,593  |  Cases: 274854  |  Controls: 335739

  Model: genes_only
      ERROR during predict_proba: "['FO5387572', 'CPSF3L', 'SLC35E2', 'FAM213B', 'TMEM57', 'ATPIF1', 'C1orf228', 'NRD1', 'ZCCHC11', 'C1orf123', 'INADL', 'ZZZ3', 'FAM73A', 'SEP15', 'CCBL2', 'KIAA1107', 'FAM69A', 'HIAT1', 'RP11475E119', 'ATP5F1', 'FAM212B', 'KIAA0907', 'RFWD2', 'C1orf27', 'TROVE2', 'MFSD4', 'C1orf95', 'ADCK3', 'PCNXL2', 'RP5862P82', 'TSSC1', 'FAM84A', 'EPT1', 'BRE', 'GPR75ASB3', 'RNF103CHMP3', 'AC0928352', 'RP11111H131', 'LINC00116', 'AC0134611', 'ATP5G3', 'KIAA1715', 'SSFA2', 'C2orf47', 'ALS2CR12', 'RQCD1', 'FAM134A', 'MYEOV2', 'SEPT2', 'GPX1', 'VPRBP', 'SELK', 'FAM208A', 'FAM19A1', 'TOMM70A', 'PVRL3', 'C3orf17', 'DIRC2', 'C3orf58', 'SELT', 'ATP5I', 'WHSC1', 'RP11231C183', 'SEPT11', 'C4orf22', 'KIAA0922', 'GUCY1A3', 'GUCY1B3', 'C4orf27', 'PAPD7', 'FAM173B', 'FAM134B', 'C1QTNF3AMA

: 